# Player and game Elo models (v2)

Train / validation / test are **player-disjoint** (assigned before aggregation). The player model estimates median Elo from behavior and Stockfish features. The game model estimates Elo from one side of one game.

**Data:** classical Lichess dump `lichess_db_standard_rated_2017-02.pgn` (Elo 800–2400). Rebuild tables per `docs/data.md`. Colab-first: put `src` under `MyDrive/src` and parquet tables under `MyDrive/DATA/`.

| Model | Table | Use |
|---|---|---|
| **Primary** | `players.parquet` | stable Elo from many games |
| **Secondary** | `games_split.parquet` | future "paste one PGN" path |

Features come from `src/schema.py` (`NUMERIC_FEATURES`, `ENGINE_FEATURES`, `GAME_EXTRA`), not from "every numeric column." `n_games` is a sample weight, not a feature. `result_mean` appears only in the ablation. Game-level `eco_family` is excluded: for Black it is mostly White's choice. `*_std` features stay out until the median player has at least 20 games.

**Protocol:** fit on train, early-stop on validation MAE, no hyperparameter grid (validation is too small to tune). Tolerance accuracy is the headline. Band accuracy is secondary. Test is scored once, in the last section, which plots predicted Elo against actual Elo for every held-out player and states what those numbers support.

**Run this in Google Colab.** Use a fresh runtime and run every cell from the top, so Drive is mounted before the training cells import anything. Colab's working directory is `/content`, not the notebook's folder.

Put the two training tables on Drive:

- `MyDrive/DATA/players.parquet`
- `MyDrive/DATA/games_split.parquet`

Models, the metric table, and the feature-order cards are written back into that same folder, so they remain after the session ends. If a Drive folder also contains `src/schema.py`, the allow-list is imported from it. Otherwise this notebook uses the copy in the imports cell.



In [ ]:
from pathlib import Path
import sys

IN_COLAB = Path("/content").exists()
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    import subprocess

    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "pyarrow>=14", "pandas>=2.0", "numpy>=1.24",
        "scikit-learn>=1.3", "scipy>=1.11", "xgboost>=2.0", "matplotlib",
    ])
    if not Path("/content/drive/MyDrive").exists():
        raise RuntimeError("Drive mount did not expose /content/drive/MyDrive")
    print("[setup] Drive mounted; dependencies installed")
    print("[setup] Expecting players.parquet and games_split.parquet in MyDrive/DATA")
    print("[setup] Fitted models will be saved back into that folder")
else:
    print("[setup] local runtime — skipping Drive mount")


In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

_NUMERIC_FEATURES = [
    "game_plies", "n_subject_moves", "rate_pawn", "rate_knight", "rate_bishop",
    "rate_rook", "rate_queen", "rate_king", "rate_capture", "rate_check",
    "rate_promotion", "castled", "castle_kingside", "castle_ply", "mat_opening_mean",
    "mat_middlegame_mean", "developed_minors_ply10", "avg_legal_moves",
]
_ENGINE_FEATURES = [
    "cpl_mean", "cpl_median", "eval_volatility", "mistake_rate", "blunder_rate", "best_move_rate",
]
_GAME_EXTRA = ["is_white"]

def _drive_dirs():
    root = Path("/content/drive/MyDrive")
    if not root.exists():
        return []
    try:
        return [path for path in root.iterdir() if path.is_dir()]
    except OSError as exc:
        print(f"[config] could not list Drive: {exc}")
        return []

def find_data():
    """Colab reads Drive first. The kernel cwd is /content, not the notebook folder."""
    in_colab = Path("/content").exists()
    if in_colab and not Path("/content/drive/MyDrive").exists():
        raise RuntimeError("Drive is not mounted. Run the setup cell and accept the Drive prompt.")
    candidates = []
    if in_colab:
        candidates.append(Path("/content/drive/MyDrive/DATA"))
        for child in _drive_dirs():
            candidates.append(child / "data")
            if child.name.lower() == "data":
                candidates.append(child)
        candidates.append(Path("/content/data"))
    candidates.extend([Path("data"), Path("../data")])
    seen = set()
    for path in candidates:
        key = str(path)
        if key in seen:
            continue
        seen.add(key)
        if (path / "players.parquet").exists() and (path / "games_split.parquet").exists():
            return path
    looked = ", ".join(str(path) for path in candidates[:8])
    raise RuntimeError(
        "Could not find players.parquet and games_split.parquet. "
        "On Colab, put both files in MyDrive/DATA/. "
        f"Looked at: {looked}"
    )

def load_allow_list():
    roots = list(_drive_dirs()) if Path("/content").exists() else []
    roots.extend([Path("/content"), Path.cwd(), Path.cwd().parent])
    seen = set()
    for root in roots:
        key = str(root)
        if key in seen:
            continue
        seen.add(key)
        schema = root / "src" / "schema.py"
        if not schema.exists():
            continue
        root_str = str(root.resolve())
        if root_str not in sys.path:
            sys.path.insert(0, root_str)
        from src.schema import ENGINE_FEATURES, GAME_EXTRA, NUMERIC_FEATURES
        print(f"[config] allow-list imported from {schema}")
        return list(NUMERIC_FEATURES), list(ENGINE_FEATURES), list(GAME_EXTRA)
    print("[config] src/schema.py not found; using the allow-list embedded in this notebook")
    return list(_NUMERIC_FEATURES), list(_ENGINE_FEATURES), list(_GAME_EXTRA)

NUMERIC_FEATURES, ENGINE_FEATURES, GAME_EXTRA = load_allow_list()
DATA = find_data()
print(f"[config] DATA -> {DATA.resolve()}")
for _filename in ("players.parquet", "games_split.parquet"):
    _file = DATA / _filename
    print(f"[config] {_filename}  {_file.stat().st_size / 1e6:.1f} MB")

def show(obj):
    try:
        display(obj)
    except NameError:
        print(obj)

SEED = 42
RATING_BINS = list(range(800, 2400 + 1, 200))
ACCURACY_TOLERANCES = (50, 100, 150, 200, 300)
N_JOBS = os.cpu_count() or 2
FROZEN_PARAMS = {
    "n_estimators": 400,
    "max_depth": 4,
    "learning_rate": 0.05,
    "min_child_weight": 10,
    "reg_lambda": 1.0,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "reg:absoluteerror",
    "tree_method": "hist",
    "early_stopping_rounds": 30,
    "random_state": SEED,
    "n_jobs": N_JOBS,
}
STD_MIN_GAMES = 20
N_GAMES_WARN = 10
MIN_BIN_TRAIN = 50
WEIGHT_CAP = 20
N_BOOT = 1000
PLAYER_EXTRA = ["opening_entropy", "first_move_entropy"]
DECREASING_FEATURES = {
    "cpl_mean", "cpl_median", "mistake_rate", "blunder_rate",
    "cpl_mean_mean", "cpl_median_mean", "mistake_rate_mean", "blunder_rate_mean",
}
INCREASING_FEATURES = {"best_move_rate", "best_move_rate_mean"}
print(f"[config] seed={SEED}  equal-width bins={RATING_BINS}")
print(
    "[config] frozen params "
    f"depth={FROZEN_PARAMS['max_depth']}  lr={FROZEN_PARAMS['learning_rate']}  "
    f"min_child_weight={FROZEN_PARAMS['min_child_weight']}  "
    f"reg_lambda={FROZEN_PARAMS['reg_lambda']}  objective={FROZEN_PARAMS['objective']}"
)


## 1. Load data and gate it

| Table | Grain | Label |
|---|---|---|
| `players` | 1 row / player | median Elo |
| `games_split` | 1 row / side / game | that game's Elo |

Both inherit the same player-level `split`, so test players never leak into training. Training stops if a split is empty, splits overlap, or a Stockfish column is missing or null. Thin rating bins and a short per-player sample print a warning and stay in the run: those are data limits, not a reason to retune.


In [ ]:
print("[load] reading parquet files...")
players = pd.read_parquet(DATA / "players.parquet")
games = pd.read_parquet(DATA / "games_split.parquet")
print(f"[load] players rows={len(players):,}  cols={players.shape[1]}")
print(f"[load] games   rows={len(games):,}  cols={games.shape[1]}")

required_splits = ("train", "val", "test")
for label, frame in (("players", players), ("games", games)):
    unknown = set(frame["split"].astype(str).unique()) - set(required_splits)
    if unknown:
        raise RuntimeError(f"{label} has unexpected split labels: {sorted(unknown)}")
    counts = frame["split"].value_counts()
    empty = [name for name in required_splits if counts.get(name, 0) == 0]
    if empty:
        raise RuntimeError(f"{label} split(s) {empty} are empty")

def split_keys(frame):
    return {name: set(frame.loc[frame["split"] == name, "player_key"]) for name in required_splits}

for label, frame in (("players", players), ("games", games)):
    keys = split_keys(frame)
    overlaps = {
        f"{a}&{b}": len(keys[a] & keys[b])
        for a, b in (("train", "val"), ("train", "test"), ("val", "test"))
    }
    if any(overlaps.values()):
        raise RuntimeError(f"{label} splits overlap: {overlaps}")
    print(f"[gate] {label} player-key overlaps={overlaps}")

player_engine = [f"{column}_mean" for column in ENGINE_FEATURES]
for label, frame, columns in (
    ("players", players, player_engine),
    ("games", games, list(ENGINE_FEATURES)),
):
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        raise RuntimeError(f"{label} is missing Stockfish columns: {missing}")
    null_cols = [column for column in columns if frame[column].isna().any()]
    if null_cols:
        raise RuntimeError(f"{label} Stockfish columns contain nulls: {null_cols}")
print(f"[gate] Stockfish columns are present and non-null ({len(player_engine)} player, {len(ENGINE_FEATURES)} game)")

meta_path = DATA / "aggregate_meta.json"
if meta_path.exists():
    STABLE_GAMES = int(json.loads(meta_path.read_text(encoding="utf-8"))["min_games"])
else:
    STABLE_GAMES = int(players["n_games"].min())
print(f"[gate] stable-player cutoff n_games>={STABLE_GAMES} (player-table minimum)")

train_players = players.loc[players["split"] == "train"].copy()
train_bins = pd.Series(pd.cut(train_players["elo"], bins=RATING_BINS, include_lowest=True))
bin_counts = train_bins.value_counts(sort=False).reindex(train_bins.cat.categories, fill_value=0)
print("\n[gate] TRAIN players per equal-width Elo bin:")
print(bin_counts.to_string())
thin = bin_counts[bin_counts < MIN_BIN_TRAIN]
if len(thin):
    print(f"[gate] WARNING: {len(thin)} train bin(s) have fewer than {MIN_BIN_TRAIN} players. Tail MAE will be weak.")
    print(thin.to_string())

median_games = float(players["n_games"].median())
INCLUDE_STD = median_games >= STD_MIN_GAMES
if median_games < N_GAMES_WARN:
    print(
        f"[gate] WARNING: median n_games={median_games:.0f} is under {N_GAMES_WARN}. "
        f"*_std features need a median of {STD_MIN_GAMES} games and stay out of the model."
    )
elif not INCLUDE_STD:
    print(f"[gate] median n_games={median_games:.0f} < {STD_MIN_GAMES}; *_std features stay out of the model.")
else:
    print(f"[gate] median n_games={median_games:.0f}; *_std features are included.")

player_ranges = games.groupby("player_key")["elo"].agg(lambda series: float(series.max() - series.min()))
NOISE_FLOOR = float(player_ranges.reindex(players["player_key"]).median())
print(
    f"[gate] within-player Elo range median={NOISE_FLOOR:.0f}. "
    "That is the label noise floor next to test MAE."
)

print("\n[split] PLAYER counts:")
player_split_counts = players["split"].value_counts()
print(player_split_counts.to_string())
print("\n[split] GAME counts (inherited from players):")
print(games["split"].value_counts().to_string())

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
order = [name for name in required_splits if name in player_split_counts.index]
axes[0].bar(order, [player_split_counts[name] for name in order], color=["#2a6fbb", "#e09f3e", "#9e2a2b"])
axes[0].set_title("Players per split")
axes[0].set_ylabel("Count")
for index, name in enumerate(order):
    axes[0].text(index, player_split_counts[name], str(player_split_counts[name]), ha="center", va="bottom")

for split_name, color in (("train", "#2a6fbb"), ("val", "#e09f3e"), ("test", "#9e2a2b")):
    subset = players.loc[players["split"] == split_name, "elo"]
    if len(subset):
        axes[1].hist(subset, bins=30, alpha=0.55, label=split_name, color=color)
axes[1].set_title("Elo distribution by split")
axes[1].set_xlabel("Median Elo")
axes[1].legend()

axes[2].hist(players["n_games"], bins=30, color="#415a77", edgecolor="white")
axes[2].set_title("Games per player (weight, not a feature)")
axes[2].set_xlabel("n_games")
fig.suptitle("Dataset overview — splits should look comparable", y=1.02)
fig.tight_layout()
plt.show()
print("[eda] Overlapping Elo histograms across splits = healthier evaluation.")


## 2. Metrics and training utilities

| Metric | Meaning | Use |
|---|---|---|
| **MAE** | average absolute Elo error | primary continuous error |
| **RMSE** | like MAE but punishes large misses | secondary |
| **R²** | variance explained vs predicting the mean | higher is better |
| **Spearman** | rank correlation | does stronger play rank higher? |
| **Accuracy @ ±T** | % of predictions within T Elo of truth | main accuracy for regression |
| **Band accuracy** | % whose predicted bin matches the true bin | secondary; equal-width bins |
| **Bias** | mean of predicted minus true, inside a bin | shrinkage toward the mean |
| **95% CI** | player bootstrap, or player-cluster bootstrap for game rows | uncertainty on MAE |

Out-of-range predictions count as band misses. They are not dropped. The loss is absolute error, matching MAE. Sample weights (`n_games`, capped) affect the player fit and early stopping only. Reported MAE counts each player once.


In [ ]:
def regression_metrics(y_true, y_pred, tolerances=ACCURACY_TOLERANCES, bins=RATING_BINS):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    abs_err = np.abs(y_pred - y_true)
    if len(y_true) > 1 and np.std(y_pred) > 0 and np.std(y_true) > 0:
        rho = float(spearmanr(y_true, y_pred).statistic)
    else:
        rho = np.nan
    metrics = {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "r2": float(r2_score(y_true, y_pred)) if len(y_true) > 1 else np.nan,
        "spearman": rho,
        "n": int(len(y_true)),
    }
    for tol in tolerances:
        metrics[f"acc_within_{tol}"] = float(np.mean(abs_err <= tol))
    true_cut = pd.cut(y_true, bins=bins, include_lowest=True)
    pred_cut = pd.cut(y_pred, bins=bins, include_lowest=True)
    true_codes = np.asarray(true_cut.codes)
    pred_codes = np.asarray(pred_cut.codes)
    known = true_codes >= 0
    if known.any():
        hit = known & (pred_codes >= 0) & (true_codes == pred_codes)
        metrics["band_accuracy"] = float(hit[known].mean())
    else:
        metrics["band_accuracy"] = np.nan
    return metrics


def bin_mae(y_true, y_pred, bins=RATING_BINS):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    labels = pd.cut(y_true, bins=bins, include_lowest=True)
    rows = []
    for label in labels.categories:
        mask = np.asarray(labels == label)
        if mask.sum() == 0:
            continue
        residual = y_pred[mask] - y_true[mask]
        rows.append({
            "bin": str(label),
            "n": int(mask.sum()),
            "mae": float(np.mean(np.abs(residual))),
            "bias": float(np.mean(residual)),
        })
    return pd.DataFrame(rows)


def bootstrap_mean_ci(values, n_boot=N_BOOT, seed=SEED):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    draws = rng.integers(0, len(values), size=(n_boot, len(values)))
    stats = values[draws].mean(axis=1)
    lo, hi = np.quantile(stats, [0.025, 0.975])
    return float(values.mean()), float(lo), float(hi)


def mae_ci(y_true, y_pred, seed=SEED):
    abs_err = np.abs(np.asarray(y_pred, dtype=float) - np.asarray(y_true, dtype=float))
    return bootstrap_mean_ci(abs_err, seed=seed)


def paired_mae_diff_ci(y_true, pred_a, pred_b, seed=SEED):
    """95% CI for MAE(pred_a) - MAE(pred_b). Negative means A is better."""
    y_true = np.asarray(y_true, dtype=float)
    diff = np.abs(np.asarray(pred_a, dtype=float) - y_true) - np.abs(np.asarray(pred_b, dtype=float) - y_true)
    return bootstrap_mean_ci(diff, seed=seed)


def clustered_mae_ci(keys, y_true, y_pred, seed=SEED):
    """Row MAE with players resampled as clusters, so repeated games are not independent."""
    abs_err = np.abs(np.asarray(y_pred, dtype=float) - np.asarray(y_true, dtype=float))
    _, inverse = np.unique(np.asarray(keys), return_inverse=True)
    sums = np.bincount(inverse, weights=abs_err)
    counts = np.bincount(inverse).astype(float)
    rng = np.random.default_rng(seed)
    draws = rng.integers(0, len(sums), size=(N_BOOT, len(sums)))
    maes = sums[draws].sum(axis=1) / counts[draws].sum(axis=1)
    lo, hi = np.quantile(maes, [0.025, 0.975])
    return float(abs_err.mean()), float(lo), float(hi)


def monotone_map(columns):
    mapping = {}
    for column in columns:
        if column in DECREASING_FEATURES:
            mapping[column] = -1
        elif column in INCREASING_FEATURES:
            mapping[column] = 1
        else:
            mapping[column] = 0
    return mapping


def player_feature_columns(include_result=False):
    columns = [f"{column}_mean" for column in [*NUMERIC_FEATURES, *ENGINE_FEATURES]]
    columns.extend(PLAYER_EXTRA)
    if INCLUDE_STD:
        columns.extend(f"{column}_std" for column in [*NUMERIC_FEATURES, *ENGINE_FEATURES])
    if include_result:
        columns.append("result_mean")
    missing = [column for column in columns if column not in players.columns]
    if missing:
        raise RuntimeError(f"player allow-list columns missing: {missing}")
    empty = [column for column in columns if not players[column].notna().any()]
    if empty:
        raise RuntimeError(f"player allow-list columns are empty: {empty}")
    return columns


def game_numeric_columns():
    columns = [*GAME_EXTRA, *NUMERIC_FEATURES, *ENGINE_FEATURES]
    missing = [column for column in columns if column not in games.columns]
    if missing:
        raise RuntimeError(f"game allow-list columns missing: {missing}")
    return columns


def n_games_weight(frame, cap=WEIGHT_CAP):
    return np.clip(frame["n_games"].to_numpy(dtype=float), 1, cap)


def fit_linear(frame, columns, weights=None):
    if frame[columns].isna().any().any():
        raise RuntimeError(f"linear baseline columns contain nulls: {columns}")
    model = LinearRegression()
    model.fit(
        frame[columns].to_numpy(dtype=float),
        frame["elo"].to_numpy(dtype=float),
        sample_weight=weights,
    )
    return model


def predict_linear(model, frame, columns):
    return model.predict(frame[columns].to_numpy(dtype=float))


def fit_xgboost(train_x, train_y, val_x, val_y, feature_names, train_w=None, val_w=None, label="model"):
    """Fit on TRAIN. Early stopping watches the last eval set, which is VAL."""
    model = XGBRegressor(**FROZEN_PARAMS, monotone_constraints=monotone_map(feature_names))
    kwargs = {
        "eval_set": [(train_x, train_y), (val_x, val_y)],
        "verbose": False,
    }
    if train_w is not None:
        if val_w is None:
            raise ValueError("val weights are required when train weights are set")
        kwargs["sample_weight"] = train_w
        kwargs["sample_weight_eval_set"] = [train_w, val_w]
    model.fit(train_x, train_y, **kwargs)
    names = list(model.get_booster().feature_names)
    if names != list(feature_names):
        raise RuntimeError(f"{label} booster feature order drifted from the allow-list")
    print(
        f"[{label}] best_iteration={model.best_iteration}  "
        f"best_score={model.best_score:.3f}  (last eval set is VAL)"
    )
    return model


def permutation_frame(model, frame, columns, n_repeats, label):
    print(f"[{label}] permutation importance on VAL ({n_repeats} repeats, MAE increase)")
    result = permutation_importance(
        model,
        frame[columns],
        frame["elo"].to_numpy(dtype=float),
        scoring="neg_mean_absolute_error",
        n_repeats=n_repeats,
        random_state=SEED,
        n_jobs=1,
    )
    table = (
        pd.DataFrame({
            "feature": list(columns),
            "importance": result.importances_mean,
            "importance_std": result.importances_std,
        })
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )
    return table


def print_metrics(tag, metrics):
    print(
        f"[{tag}] n={metrics['n']:,}  "
        f"MAE={metrics['mae']:.2f}  RMSE={metrics['rmse']:.2f}  "
        f"R2={metrics['r2']:.3f}  Spearman={metrics['spearman']}"
    )
    acc_parts = [
        f"±{tol}={100 * metrics[f'acc_within_{tol}']:.1f}%"
        for tol in ACCURACY_TOLERANCES
        if f"acc_within_{tol}" in metrics
    ]
    band = metrics.get("band_accuracy", np.nan)
    band_txt = f"{100 * band:.1f}%" if band == band else "nan"
    print(f"[{tag}] accuracy  " + "  ".join(acc_parts) + f"  |  band_accuracy(secondary)={band_txt}")


def print_interval(tag, point, lo, hi, how):
    print(f"[{tag}] MAE={point:.2f}  95% CI [{lo:.1f}, {hi:.1f}]  ({how})")


def plot_accuracy_bars(metric_rows, title="Accuracy within Elo tolerance"):
    frame = pd.DataFrame(metric_rows)
    tol_cols = [f"acc_within_{tol}" for tol in ACCURACY_TOLERANCES]
    plot_df = frame.set_index("model")[tol_cols] * 100
    plot_df.columns = [f"±{tol}" for tol in ACCURACY_TOLERANCES]
    ax = plot_df.T.plot(kind="bar", figsize=(8.5, 4.2), rot=0)
    ax.set_ylabel("Accuracy (%)")
    ax.set_xlabel("Tolerance around true Elo")
    ax.set_ylim(0, 100)
    ax.set_title(title)
    ax.legend(title="Model", fontsize=8)
    fig = ax.get_figure()
    fig.tight_layout()
    plt.show()
    return plot_df


def plot_diagnostics(y_true, y_pred, title, importance=None, sample_cap=4000):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    residuals = y_pred - y_true
    bins = bin_mae(y_true, y_pred)

    rng = np.random.default_rng(SEED)
    if len(y_true) > sample_cap:
        idx = rng.choice(len(y_true), size=sample_cap, replace=False)
        plot_true, plot_pred = y_true[idx], y_pred[idx]
    else:
        plot_true, plot_pred = y_true, y_pred

    n_cols = 3 if importance is not None else 2
    fig, axes = plt.subplots(1, n_cols, figsize=(5.4 * n_cols, 4.3))
    if n_cols == 2:
        axes = list(axes)

    lo = float(min(plot_true.min(), plot_pred.min()))
    hi = float(max(plot_true.max(), plot_pred.max()))
    axes[0].scatter(plot_true, plot_pred, s=14, alpha=0.45, edgecolors="none", c="#2a6fbb")
    axes[0].plot([lo, hi], [lo, hi], color="black", lw=1.2, label="perfect calibration")
    axes[0].set_xlabel("True Elo")
    axes[0].set_ylabel("Predicted Elo")
    axes[0].set_title(f"{title}: pred vs true")
    axes[0].legend(loc="upper left", fontsize=8)
    axes[0].set_aspect("equal", adjustable="box")

    axes[1].bar(bins["bin"].astype(str), bins["mae"], color="#415a77")
    axes[1].tick_params(axis="x", rotation=35)
    axes[1].set_ylabel("MAE (Elo)")
    axes[1].set_title("MAE by true-Elo bin (bias = pred - true)")
    for index, row in bins.iterrows():
        axes[1].text(
            index, row["mae"], f"n={row['n']}\nbias={row['bias']:+.0f}",
            ha="center", va="bottom", fontsize=7,
        )

    if importance is not None:
        top = importance.head(12).iloc[::-1]
        axes[2].barh(top["feature"], top["importance"], color="#e09f3e")
        axes[2].set_title("Permutation importance (VAL, MAE increase)")
        axes[2].set_xlabel("MAE increase when shuffled")

    fig.suptitle(
        f"{title}  |  residual mean={residuals.mean():.1f}, std={residuals.std():.1f}",
        y=1.02,
        fontsize=11,
    )
    fig.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(6.2, 3.6))
    ax.hist(residuals, bins=40, color="#778da9", edgecolor="white")
    ax.axvline(0, color="black", lw=1.2)
    ax.set_xlabel("Predicted - true Elo")
    ax.set_ylabel("Count")
    ax.set_title(f"{title}: residual distribution (want centered near 0)")
    fig.tight_layout()
    plt.show()
    print(f"[plot] {title}: diagonal scatter, bin MAE with bias, residuals near 0.")
    return bins

def plot_pred_vs_actual(y_true, y_pred, title, tolerance=150):
    """Held-out calibration. The shaded band is the tolerance used as accuracy."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    residual = y_pred - y_true
    mae = float(np.mean(np.abs(residual)))
    lo = float(min(y_true.min(), y_pred.min())) - 40
    hi = float(max(y_true.max(), y_pred.max())) + 40

    labels = pd.cut(y_true, bins=RATING_BINS, include_lowest=True)
    centers, pred_means, resid_means, counts = [], [], [], []
    for interval in labels.categories:
        mask = np.asarray(labels == interval)
        if not mask.any():
            continue
        centers.append(float(y_true[mask].mean()))
        pred_means.append(float(y_pred[mask].mean()))
        resid_means.append(float(residual[mask].mean()))
        counts.append(int(mask.sum()))

    fig, axes = plt.subplots(1, 3, figsize=(14.8, 4.8))
    ax = axes[0]
    span = np.array([lo, hi])
    ax.fill_between(
        span, span - tolerance, span + tolerance,
        color="#2a6fbb", alpha=0.12, linewidth=0, label=f"within {tolerance} Elo",
    )
    dense = len(y_true) > 2500
    if dense:
        bins_hb = ax.hexbin(y_true, y_pred, gridsize=32, cmap="Blues", mincnt=1)
        fig.colorbar(bins_hb, ax=ax, fraction=0.046, pad=0.04, label="rows")
    else:
        ax.scatter(y_true, y_pred, s=36, alpha=0.85, c="#1d3557", edgecolors="none", zorder=3)
    ax.plot([lo, hi], [lo, hi], color="black", lw=1.3, label="predicted = actual")
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Actual Elo")
    ax.set_ylabel("Predicted Elo")
    ax.set_title("Predicted vs actual")
    ax.legend(loc="upper left", fontsize=8)

    ax = axes[1]
    ax.plot([lo, hi], [lo, hi], color="black", lw=1.3, label="predicted = actual")
    ax.scatter(centers, pred_means, s=[20 + 3 * n for n in counts], c="#9e2a2b", zorder=3)
    for x, y, n in zip(centers, pred_means, counts):
        ax.annotate(str(n), (x, y), textcoords="offset points", xytext=(5, 4), fontsize=8)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Mean actual Elo in bin")
    ax.set_ylabel("Mean predicted Elo in bin")
    ax.set_title("Calibration (annotation = n)")

    ax = axes[2]
    ax.axhline(0, color="black", lw=1.2)
    if dense:
        take = np.random.default_rng(SEED).choice(len(y_true), size=2000, replace=False)
        ax.scatter(y_true[take], residual[take], s=10, alpha=0.3, c="#415a77", edgecolors="none")
    else:
        ax.scatter(y_true, residual, s=36, alpha=0.85, c="#415a77", edgecolors="none")
    ax.plot(centers, resid_means, color="#9e2a2b", marker="o", lw=1.6, label="mean error in bin")
    ax.set_xlabel("Actual Elo")
    ax.set_ylabel("Predicted - actual")
    ax.set_title("Error along the rating scale")
    ax.legend(fontsize=8)

    fig.suptitle(f"{title}   |   test MAE {mae:.0f}   |   n={len(y_true):,}", fontsize=12, y=1.03)
    fig.tight_layout()
    plt.show()
    return bin_mae(y_true, y_pred)


def state_findings(
    player_mae, player_lo, player_hi, spearman, acc150,
    gap, gap_lo, gap_hi, baseline_name, earned,
    abl_gap, abl_lo, abl_hi, leak,
    noise_floor, player_bins,
    game_row_mae, game_collapsed_mae, game_earned, game_lo, game_hi,
):
    print("\n[findings] ===== WHAT THE TEST RESULTS SUPPORT =====")
    if earned:
        print(
            f"[findings] The player tree is the model to report. It lowers test MAE by {-gap:.0f} Elo "
            f"versus the {baseline_name} baseline. The 95% interval on tree minus baseline "
            f"[{gap_lo:.0f}, {gap_hi:.0f}] lies entirely below zero."
        )
    else:
        print(
            f"[findings] Report the {baseline_name} baseline, not the tree. The MAE gap is {gap:+.0f} Elo "
            f"with 95% interval [{gap_lo:.0f}, {gap_hi:.0f}], which does not sit entirely below zero. "
            "The extra behavior features are not justified on this test set."
        )
    if player_mae <= noise_floor + 50:
        print(
            f"[findings] Player test MAE is {player_mae:.0f} (95% CI [{player_lo:.0f}, {player_hi:.0f}]). "
            f"A player's Elo already moves about {noise_floor:.0f} points across the games in this extract, "
            "so this error is in the range of the label itself."
        )
    else:
        print(
            f"[findings] Player test MAE is {player_mae:.0f} (95% CI [{player_lo:.0f}, {player_hi:.0f}]), "
            f"larger than the {noise_floor:.0f}-point movement in a player's own Elo. "
            "Error remains beyond label noise."
        )
    if spearman >= 0.7:
        rank = "strong enough to order a pool of players"
    elif spearman >= 0.5:
        rank = "real but only moderate"
    else:
        rank = "too weak to treat the scores as a ranking"
    print(f"[findings] Spearman {spearman:.2f} is {rank}.")
    if acc150 >= 0.6:
        place = "A majority of held-out players land inside that band."
    else:
        place = "Most held-out players fall outside that band, so the point prediction is a coarse placement."
    print(f"[findings] {100 * acc150:.0f}% of test players are within 150 Elo of their actual rating. {place}")
    if len(player_bins):
        low = player_bins.iloc[0]
        high = player_bins.iloc[-1]
        if low["bias"] > 40 and high["bias"] < -40:
            print(
                f"[findings] The calibration plot shrinks toward the mean: bin {low['bin']} "
                f"(n={low['n']}) is over-predicted by {low['bias']:.0f} Elo, and bin {high['bin']} "
                f"(n={high['n']}) is under-predicted by {abs(high['bias']):.0f}. "
                "Do not treat tail predictions as identified ratings."
            )
        elif abs(low["bias"]) <= 40 and abs(high["bias"]) <= 40:
            print(
                "[findings] Mean error stays within 40 Elo at both ends of the rating scale, "
                "so the test predictions are calibrated rather than shrunk to the mean."
            )
        else:
            print(
                f"[findings] Bias is uneven across the scale: {low['bias']:+.0f} Elo in {low['bin']} "
                f"(n={low['n']}) and {high['bias']:+.0f} Elo in {high['bin']} (n={high['n']})."
            )
    if leak:
        print(
            f"[findings] Win rate lowers test MAE by {-abl_gap:.0f} Elo and the interval "
            f"[{abl_lo:.0f}, {abl_hi:.0f}] lies entirely below zero. Outcomes are standing in for "
            "opponent strength. Do not ship the ablation model."
        )
    else:
        print(
            f"[findings] Win rate changes test MAE by {abl_gap:+.1f} Elo "
            f"(95% CI [{abl_lo:.1f}, {abl_hi:.1f}]). The interval crosses zero, so game results "
            "are not required for the estimate."
        )
    if game_earned:
        game_line = (
            f"The single-game tree also beats its linear baseline; the cluster interval "
            f"[{game_lo:.0f}, {game_hi:.0f}] lies below zero."
        )
    else:
        game_line = (
            f"The single-game tree does not clear its linear baseline "
            f"(cluster interval [{game_lo:.0f}, {game_hi:.0f}])."
        )
    print(
        f"[findings] {game_line} One side of one game has MAE {game_row_mae:.0f}. "
        f"Averaging a player's test games brings that to {game_collapsed_mae:.0f}. "
        "Use the player model when several games are available."
    )



_band_check = regression_metrics(np.array([900.0, 1500.0]), np.array([500.0, 1500.0]))
if abs(_band_check["band_accuracy"] - 0.5) > 1e-9:
    raise RuntimeError(f"band accuracy dropped an out-of-range prediction: {_band_check}")
print("[utils] metric + plotting helpers ready; band-accuracy self-check passed")


## 3. Player-level model (primary)

**Target:** median Elo. **Fit:** train, with sample weights `min(n_games, 20)`. **Early stop:** validation MAE. **Config:** frozen (`max_depth` 4, `min_child_weight` 10, `reg_lambda` 1, absolute error). There is no grid.

**Baselines, fit on the same rows:**

1. Univariate linear regression on `mistake_rate_mean`.
2. Linear regression on the Stockfish block (`cpl_*`, `mistake_rate`, `blunder_rate`, `best_move_rate`).

The mean predictor stays in the final table as a weak reference. XGBoost earns its place only if its test MAE sits below the better linear baseline by more than the paired bootstrap allows. That comparison is in the last cell. This cell reports validation only.


In [ ]:
print("[player] ===== ALLOW-LIST =====")
player_cols = player_feature_columns(include_result=False)
engine_cols = [f"{column}_mean" for column in ENGINE_FEATURES]
uni_col = "mistake_rate_mean"
if not INCLUDE_STD:
    print(
        f"[player] *_std dropped (median n_games={players['n_games'].median():.0f} < {STD_MIN_GAMES}). "
        "Collect more games per player before adding them back."
    )
else:
    print(f"[player] *_std included (median n_games={players['n_games'].median():.0f})")
print(f"[player] features={len(player_cols)}")
print("[player] " + ", ".join(player_cols))
null_rate = players[player_cols].isna().mean()
null_rate = null_rate[null_rate > 0]
if len(null_rate):
    print("[player] allow-list null rates (kept as XGBoost missing values):")
    print(null_rate.round(3).to_string())

train_players = players.loc[players["split"] == "train"].copy()
val_players = players.loc[players["split"] == "val"].copy()
w_train = n_games_weight(train_players)
w_val = n_games_weight(val_players)
print(
    f"[player] TRAIN n={len(train_players):,}  mean_elo={train_players['elo'].mean():.1f}  "
    f"weight median={np.median(w_train):.0f}"
)
print(f"[player] VAL   n={len(val_players):,}  mean_elo={val_players['elo'].mean():.1f}")
print("[player] TEST stays sealed until the final cell")

print("\n[player] ===== BASELINES (weighted fit, unweighted VAL MAE) =====")
uni_model = fit_linear(train_players, [uni_col], w_train)
engine_model = fit_linear(train_players, engine_cols, w_train)
uni_val = predict_linear(uni_model, val_players, [uni_col])
engine_val = predict_linear(engine_model, val_players, engine_cols)
mean_val = np.full(len(val_players), float(train_players["elo"].mean()))
val_mean_metrics = regression_metrics(val_players["elo"], mean_val)
val_uni_metrics = regression_metrics(val_players["elo"], uni_val)
val_engine_metrics = regression_metrics(val_players["elo"], engine_val)
print_metrics("player/mean/VAL", val_mean_metrics)
print_metrics("player/univariate/VAL", val_uni_metrics)
print_metrics("player/engine-linear/VAL", val_engine_metrics)
VAL_LINEAR_MAE = {
    "univariate mistake_rate": val_uni_metrics["mae"],
    "engine linear": val_engine_metrics["mae"],
}
best_linear_name = min(VAL_LINEAR_MAE, key=VAL_LINEAR_MAE.get)
print(f"[player] better linear baseline on VAL: {best_linear_name}  MAE={VAL_LINEAR_MAE[best_linear_name]:.2f}")

print("\n[player] ===== XGBOOST (frozen config) =====")
player_model = fit_xgboost(
    train_players[player_cols],
    train_players["elo"].to_numpy(dtype=float),
    val_players[player_cols],
    val_players["elo"].to_numpy(dtype=float),
    player_cols,
    train_w=w_train,
    val_w=w_val,
    label="player",
)
player_val_pred = player_model.predict(val_players[player_cols])
val_player_metrics = regression_metrics(val_players["elo"], player_val_pred)
print_metrics("player/xgb/VAL", val_player_metrics)
print(
    f"[player] VAL MAE vs {best_linear_name}: "
    f"{VAL_LINEAR_MAE[best_linear_name] - val_player_metrics['mae']:+.2f} Elo "
    "(positive means XGBoost is lower). Test decides whether that gap is real."
)


In [ ]:
print("[player] ===== VALIDATION DIAGNOSTICS =====")
val_view = players.loc[players["split"] == "val"].copy()
val_view["pred"] = player_model.predict(val_view[player_cols])
val_view["residual"] = val_view["pred"] - val_view["elo"]
val_view["abs_err"] = val_view["residual"].abs()

player_importance = permutation_frame(player_model, val_view, player_cols, n_repeats=10, label="player")
print("[player] top permutation importances (VAL MAE increase):")
show(player_importance.head(20).round(3))

player_bins = plot_diagnostics(
    val_view["elo"], val_view["pred"], "Player / VAL", player_importance,
)
print("[player] stratified VAL MAE and bias (pred - true):")
show(player_bins.round(2))

try:
    evals = player_model.evals_result()
    fig, ax = plt.subplots(figsize=(6.5, 3.8))
    ax.plot(evals["validation_0"]["mae"], label="TRAIN MAE", color="#2a6fbb")
    ax.plot(evals["validation_1"]["mae"], label="VAL MAE", color="#9e2a2b")
    ax.set_xlabel("Boosting round")
    ax.set_ylabel("MAE")
    ax.set_title("Player model: TRAIN vs VAL learning curves")
    ax.legend()
    fig.tight_layout()
    plt.show()
    print("[player] Curves should fall then flatten. A wide TRAIN<<VAL gap suggests overfitting.")
except Exception as exc:
    print(f"[player] learning curve unavailable: {exc}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(val_view["elo"], val_view["residual"], s=22, alpha=0.65, c="#9e2a2b")
axes[0].axhline(0, color="black", lw=1)
axes[0].set_xlabel("True Elo")
axes[0].set_ylabel("Residual (pred - true)")
axes[0].set_title("Residuals vs true Elo (VAL)")
axes[1].scatter(val_view["n_games"], val_view["abs_err"], s=22, alpha=0.65, c="#415a77")
axes[1].set_xlabel("n_games")
axes[1].set_ylabel("|error|")
axes[1].set_title("|error| vs sample size (VAL)")
fig.tight_layout()
plt.show()

residual_cols = [
    "player_key", "elo", "pred", "residual", "n_games",
    "mistake_rate_mean", "blunder_rate_mean", "cpl_mean_mean",
]
residual_cols = [column for column in residual_cols if column in val_view.columns]
print("[player] largest VAL residuals (not a license to retune on test):")
show(val_view.nlargest(10, "abs_err")[residual_cols].round(2))


### How to read the figures

Validation plots check the fit before test is opened. The reported figure is the test calibration in the last section.

- **Predicted vs actual:** the diagonal is a perfect estimate. The band is ±150 Elo. A round cloud on the diagonal means the model has collapsed to the mean.
- **Calibration:** each point is the mean prediction inside a true-Elo bin. The annotation is the number of test players. A point left of the diagonal is over-predicted.
- **Error along the scale:** a line that falls from positive to negative is shrinkage toward the average rating. Tail bins with few players cannot support a separate claim.
- **Permutation importance:** increase in validation MAE when that column is shuffled.


## 4. Result ablation (leakage check)

Retrain with `result_mean` (win rate) added to the same allow-list and the same frozen config. A lower MAE can mean outcomes are standing in for opponent strength.

The decision rule is **not** a fixed 5 Elo gap. In the final cell, a paired bootstrap over test players calls leakage only when the win-rate model's MAE interval sits entirely below the behavior model.


In [ ]:
print("[ablation] ===== ALLOW-LIST + result_mean =====")
ablation_cols = player_feature_columns(include_result=True)
print(f"[ablation] features={len(ablation_cols)} (behavior had {len(player_cols)})")
ablation_model = fit_xgboost(
    train_players[ablation_cols],
    train_players["elo"].to_numpy(dtype=float),
    val_players[ablation_cols],
    val_players["elo"].to_numpy(dtype=float),
    ablation_cols,
    train_w=w_train,
    val_w=w_val,
    label="ablation",
)
ablation_val_pred = ablation_model.predict(val_players[ablation_cols])
val_ablation_metrics = regression_metrics(val_players["elo"], ablation_val_pred)
print_metrics("ablation/xgb/VAL", val_ablation_metrics)
print_metrics("behavior/xgb/VAL", val_player_metrics)
print(
    "[ablation] VAL MAE(ablation) - MAE(behavior) = "
    f"{val_ablation_metrics['mae'] - val_player_metrics['mae']:+.2f}. "
    "Pass/fail waits for the paired test interval."
)

fig, ax = plt.subplots(figsize=(5.5, 3.8))
labels = ["behavior\n(VAL)", "ablation\n(+win rate, VAL)"]
vals = [val_player_metrics["mae"], val_ablation_metrics["mae"]]
ax.bar(labels, vals, color=["#2a6fbb", "#2a9d8f"])
ax.set_ylabel("VAL MAE (Elo)")
ax.set_title("Ablation on validation (test is later)")
for index, value in enumerate(vals):
    ax.text(index, value, f"{value:.1f}", ha="center", va="bottom")
fig.tight_layout()
plt.show()


## 5. Game-level model (single-match path)

Same player-disjoint splits and the same frozen tree. Features are the schema allow-list plus `is_white`, and a train-only one-hot of the **subject's** `first_move`. `eco_family` is not used.

Validation metrics below are row-level. The final cell also reports:

- a player-cluster bootstrap, because rows from the same player are not independent
- one prediction per player (mean prediction vs that player's median Elo)
- the `n_games >=` player-table cutoff separately from one-off players


In [ ]:
print("[game] ===== ALLOW-LIST (no eco_family) =====")
game_numeric_cols = game_numeric_columns()
if "eco_family" in game_numeric_cols:
    raise RuntimeError("eco_family leaked into the game allow-list")
print(f"[game] numeric features={len(game_numeric_cols)}")
print("[game] " + ", ".join(game_numeric_cols))

prepared = {}
games_per_player = games.groupby("player_key").size()
for name in ("train", "val", "test"):
    group = games.loc[games["split"] == name].reset_index(drop=True)
    group["n_games_player"] = group["player_key"].map(games_per_player).to_numpy()
    prepared[name] = group
    print(f"[game] split={name:5s}  rows={len(group):,}  players={group['player_key'].nunique():,}")

game_encoder = OneHotEncoder(handle_unknown="ignore", max_categories=30, sparse_output=False)
print(f"[game] fitting OneHotEncoder on TRAIN first_move only (n={len(prepared['train']):,})")
game_encoder.fit(prepared["train"][["first_move"]].fillna("missing").astype(str))

def game_design(group):
    numeric = group[game_numeric_cols].reset_index(drop=True)
    encoded = pd.DataFrame(
        game_encoder.transform(group[["first_move"]].fillna("missing").astype(str)),
        columns=list(game_encoder.get_feature_names_out(["first_move"])),
    )
    return pd.concat([numeric, encoded], axis=1)

game_X = {name: game_design(prepared[name]) for name in ("train", "val", "test")}
game_feature_names = list(game_X["train"].columns)
print(f"[game] design width={len(game_feature_names)} (numeric + first_move one-hot)")

print("\n[game] ===== BASELINES on VAL =====")
game_uni = fit_linear(prepared["train"], ["mistake_rate"])
game_engine = fit_linear(prepared["train"], list(ENGINE_FEATURES))
game_train_mean = float(prepared["train"]["elo"].mean())
game_mean_val = np.full(len(prepared["val"]), game_train_mean)
game_uni_val = predict_linear(game_uni, prepared["val"], ["mistake_rate"])
game_engine_val = predict_linear(game_engine, prepared["val"], list(ENGINE_FEATURES))
print_metrics("game/mean/VAL", regression_metrics(prepared["val"]["elo"], game_mean_val))
print_metrics("game/univariate/VAL", regression_metrics(prepared["val"]["elo"], game_uni_val))
print_metrics("game/engine-linear/VAL", regression_metrics(prepared["val"]["elo"], game_engine_val))
GAME_VAL_LINEAR_MAE = {
    "univariate mistake_rate": float(mean_absolute_error(prepared["val"]["elo"], game_uni_val)),
    "engine linear": float(mean_absolute_error(prepared["val"]["elo"], game_engine_val)),
}
best_game_linear = min(GAME_VAL_LINEAR_MAE, key=GAME_VAL_LINEAR_MAE.get)
print(f"[game] better linear baseline on VAL: {best_game_linear}")

print("\n[game] ===== XGBOOST (frozen config, unweighted rows) =====")
game_model = fit_xgboost(
    game_X["train"],
    prepared["train"]["elo"].to_numpy(dtype=float),
    game_X["val"],
    prepared["val"]["elo"].to_numpy(dtype=float),
    game_feature_names,
    label="game",
)
game_val_pred = game_model.predict(game_X["val"])
val_game_metrics = regression_metrics(prepared["val"]["elo"], game_val_pred)
print_metrics("game/xgb/VAL rows", val_game_metrics)

game_importance = permutation_frame(
    game_model, game_X["val"].assign(elo=prepared["val"]["elo"].to_numpy()),
    game_feature_names, n_repeats=5, label="game",
)
print("[game] top permutation importances (VAL row MAE increase):")
show(game_importance.head(15).round(3))
game_bins = plot_diagnostics(prepared["val"]["elo"], game_val_pred, "Game / VAL", game_importance)
show(game_bins.round(2))


## 6. Test evaluation

The models above were fit on train and early-stopped on validation. This section scores them on test once, plots predicted Elo against actual Elo, and writes the conclusion those numbers support.

Player intervals resample players. Game-row intervals resample `player_key`. The collapsed game score compares each player's mean prediction with that player's median Elo.

The tree is the reported player model only when its paired interval against the better linear baseline lies entirely below zero. Win rate is a leak only when its interval lies entirely below the behavior model.


In [ ]:
print("[test] ===== PLAYER TEST (scored once) =====")
test_players = players.loc[players["split"] == "test"].copy()
y_test = test_players["elo"].to_numpy(dtype=float)
mean_test = np.full(len(test_players), float(train_players["elo"].mean()))
uni_test = predict_linear(uni_model, test_players, [uni_col])
engine_test = predict_linear(engine_model, test_players, engine_cols)
player_test = player_model.predict(test_players[player_cols])
ablation_test = ablation_model.predict(test_players[ablation_cols])

player_rows = []
for name, pred in (
    ("player mean", mean_test),
    ("player univariate", uni_test),
    ("player engine linear", engine_test),
    ("player xgboost", player_test),
    ("player + win-rate ablation", ablation_test),
):
    metrics = regression_metrics(y_test, pred)
    point, lo, hi = mae_ci(y_test, pred)
    metrics.update({"model": name, "grain": "player", "mae_ci_lo": lo, "mae_ci_hi": hi})
    player_rows.append(metrics)
    print_metrics(f"{name}/TEST", metrics)
    print_interval(f"{name}/TEST", point, lo, hi, "player bootstrap")

print(
    f"[test] within-player Elo range median={NOISE_FLOOR:.0f}. "
    "Read mid-band MAE against that moving label, not against zero."
)
linear_test = {
    "univariate mistake_rate": uni_test,
    "engine linear": engine_test,
}[best_linear_name]
gap, gap_lo, gap_hi = paired_mae_diff_ci(y_test, player_test, linear_test, seed=SEED + 1)
xgb_earned = gap_hi < 0
print(
    f"[test] MAE(xgb) - MAE({best_linear_name}) = {gap:+.2f}  "
    f"95% CI [{gap_lo:.1f}, {gap_hi:.1f}]"
)
if xgb_earned:
    print("[test] XGBoost earns its place: the interval sits below zero.")
else:
    print("[test] XGBoost does not clear the linear baseline: the interval does not sit below zero.")

abl_gap, abl_lo, abl_hi = paired_mae_diff_ci(y_test, ablation_test, player_test, seed=SEED + 2)
leak = abl_hi < 0
print(
    f"[test] MAE(ablation) - MAE(behavior) = {abl_gap:+.2f}  "
    f"95% CI [{abl_lo:.1f}, {abl_hi:.1f}]"
)
if leak:
    print("[test] LEAK RISK — the win-rate MAE interval sits entirely below the behavior model.")
else:
    print("[test] OK — win rate does not clearly beat behavior once uncertainty is counted.")

print("\n[test] ===== PLAYER TEST CALIBRATION =====")
player_test_bins = plot_pred_vs_actual(y_test, player_test, "Player XGBoost — test")
plot_pred_vs_actual(y_test, linear_test, f"Player baseline ({best_linear_name}) — test")
show(player_test_bins.round(1))

print("\n[test] ===== GAME TEST (scored once) =====")
test_games = prepared["test"].copy()
test_games["pred"] = game_model.predict(game_X["test"])
game_mean_test = np.full(len(test_games), game_train_mean)
game_uni_test = predict_linear(game_uni, test_games, ["mistake_rate"])
game_engine_test = predict_linear(game_engine, test_games, list(ENGINE_FEATURES))

def add_game_row(rows, name, pred, frame):
    metrics = regression_metrics(frame["elo"], pred)
    point, lo, hi = clustered_mae_ci(frame["player_key"], frame["elo"], pred)
    metrics.update({"model": name, "grain": "game_row", "mae_ci_lo": lo, "mae_ci_hi": hi})
    rows.append(metrics)
    print_metrics(f"{name}/TEST rows", metrics)
    print_interval(f"{name}/TEST rows", point, lo, hi, "player-cluster bootstrap")
    return metrics

game_rows = []
add_game_row(game_rows, "game mean", game_mean_test, test_games)
add_game_row(game_rows, "game univariate", game_uni_test, test_games)
add_game_row(game_rows, "game engine linear", game_engine_test, test_games)
game_row_metrics = add_game_row(game_rows, "game xgboost", test_games["pred"], test_games)

game_linear_test = {
    "univariate mistake_rate": game_uni_test,
    "engine linear": game_engine_test,
}[best_game_linear]
game_gap, game_gap_lo, game_gap_hi = paired_mae_diff_ci(
    test_games["elo"], test_games["pred"], game_linear_test, seed=SEED + 3,
)
row_diff = np.abs(test_games["pred"] - test_games["elo"]) - np.abs(game_linear_test - test_games["elo"])
_, inverse = np.unique(test_games["player_key"].to_numpy(), return_inverse=True)
diff_sums = np.bincount(inverse, weights=row_diff)
diff_counts = np.bincount(inverse).astype(float)
rng = np.random.default_rng(SEED + 4)
draws = rng.integers(0, len(diff_sums), size=(N_BOOT, len(diff_sums)))
cluster_gaps = diff_sums[draws].sum(axis=1) / diff_counts[draws].sum(axis=1)
cg_lo, cg_hi = np.quantile(cluster_gaps, [0.025, 0.975])
game_xgb_earned = float(cg_hi) < 0
print(
    f"[test] game MAE(xgb) - MAE({best_game_linear}) = {game_gap:+.2f}  "
    f"row-bootstrap CI [{game_gap_lo:.1f}, {game_gap_hi:.1f}]  "
    f"cluster CI [{float(cg_lo):.1f}, {float(cg_hi):.1f}]"
)
if game_xgb_earned:
    print("[test] Game XGBoost earns its place on the cluster interval.")
else:
    print("[test] Game XGBoost does not clear the linear baseline on the cluster interval.")

collapsed = (
    test_games.groupby("player_key", sort=False)
    .agg(elo=("elo", "median"), pred=("pred", "mean"), n_games_player=("n_games_player", "first"))
    .reset_index()
)
collapsed_metrics = regression_metrics(collapsed["elo"], collapsed["pred"])
c_point, c_lo, c_hi = mae_ci(collapsed["elo"], collapsed["pred"], seed=SEED + 5)
collapsed_metrics.update({
    "model": "game xgboost collapsed",
    "grain": "game_player",
    "mae_ci_lo": c_lo,
    "mae_ci_hi": c_hi,
})
print_metrics("game/xgb/TEST one row per player", collapsed_metrics)
print_interval("game/xgb/TEST collapsed", c_point, c_lo, c_hi, "player bootstrap")
print("\n[test] ===== GAME TEST CALIBRATION =====")
plot_pred_vs_actual(
    collapsed["elo"], collapsed["pred"],
    "Game XGBoost, one prediction per test player",
)
plot_pred_vs_actual(
    test_games["elo"], test_games["pred"],
    "Game XGBoost, one row per side",
)

slice_rows = []
for slice_name, mask in (
    (f"n_games>={STABLE_GAMES}", test_games["n_games_player"] >= STABLE_GAMES),
    (f"n_games<{STABLE_GAMES}", test_games["n_games_player"] < STABLE_GAMES),
):
    part = test_games.loc[mask]
    if part.empty:
        print(f"[test] slice {slice_name} is empty")
        continue
    sliced = regression_metrics(part["elo"], part["pred"])
    s_point, s_lo, s_hi = clustered_mae_ci(part["player_key"], part["elo"], part["pred"])
    sliced.update({
        "model": f"game xgboost {slice_name}",
        "grain": "game_row",
        "mae_ci_lo": s_lo,
        "mae_ci_hi": s_hi,
    })
    slice_rows.append(sliced)
    print_metrics(f"game/xgb/TEST {slice_name}", sliced)
    print_interval(f"game/xgb/TEST {slice_name}", s_point, s_lo, s_hi, "player-cluster bootstrap")
    collapsed_part = (
        part.groupby("player_key", sort=False)
        .agg(elo=("elo", "median"), pred=("pred", "mean"))
        .reset_index()
    )
    collapsed_slice = regression_metrics(collapsed_part["elo"], collapsed_part["pred"])
    cs_point, cs_lo, cs_hi = mae_ci(collapsed_part["elo"], collapsed_part["pred"], seed=SEED + 6)
    collapsed_slice.update({
        "model": f"game xgboost collapsed {slice_name}",
        "grain": "game_player",
        "mae_ci_lo": cs_lo,
        "mae_ci_hi": cs_hi,
    })
    slice_rows.append(collapsed_slice)
    print_metrics(f"game/xgb/TEST collapsed {slice_name}", collapsed_slice)
    print_interval(f"game/xgb/TEST collapsed {slice_name}", cs_point, cs_lo, cs_hi, "player bootstrap")

print("\n[summary] ===== FINAL METRICS TABLE =====")
summary = pd.DataFrame([*player_rows, *game_rows, collapsed_metrics, *slice_rows])
acc_cols = [
    "model", "grain", "n", "mae", "mae_ci_lo", "mae_ci_hi", "rmse", "r2", "spearman", "band_accuracy",
] + [f"acc_within_{tol}" for tol in ACCURACY_TOLERANCES]
show(summary[acc_cols].round(3))

headline = next(row for row in player_rows if row["model"] == "player xgboost")
print(
    "[summary] player XGBoost test accuracy @ ±150: "
    f"{100 * headline['acc_within_150']:.1f}%  |  "
    f"band accuracy (secondary): {100 * headline['band_accuracy']:.1f}%  |  "
    f"MAE 95% CI [{headline['mae_ci_lo']:.1f}, {headline['mae_ci_hi']:.1f}]"
)
print(
    "[summary] game XGBoost row accuracy @ ±150: "
    f"{100 * game_row_metrics['acc_within_150']:.1f}%  |  "
    f"collapsed MAE={collapsed_metrics['mae']:.1f}"
)

fig, ax = plt.subplots(figsize=(8.5, 5.2))
plot_df = summary.set_index("model")["mae"]
plot_df.plot(kind="barh", ax=ax, color="#415a77")
ax.set_xlabel("TEST MAE (Elo)")
ax.set_title("Held-out TEST MAE (lower is better)")
fig.tight_layout()
plt.show()

plot_accuracy_bars(
    [
        {"model": "player mean", **player_rows[0]},
        {"model": "player univariate", **player_rows[1]},
        {"model": "player engine linear", **player_rows[2]},
        {"model": "player xgboost", **player_rows[3]},
        {"model": "game xgboost", **game_row_metrics},
    ],
    title="TEST accuracy within Elo tolerance (higher better)",
)

def linear_card(model, columns):
    return {
        "features": list(columns),
        "coef": [float(value) for value in np.ravel(model.coef_)],
        "intercept": float(model.intercept_),
    }

out_dir = DATA if DATA.exists() else Path(".")
player_model.save_model(out_dir / "player_xgb.json")
game_model.save_model(out_dir / "game_xgb.json")
player_card = {
    "model": "player",
    "primary": True,
    "xgb_earned": bool(xgb_earned),
    "comparison_baseline": best_linear_name,
    "mae_gap_xgb_minus_baseline": gap,
    "mae_gap_ci95": [gap_lo, gap_hi],
    "ablation_gap_vs_behavior": abl_gap,
    "ablation_gap_ci95": [abl_lo, abl_hi],
    "ablation_leak": bool(leak),
    "protocol": "fit train, early-stop on val MAE, no grid, test scored once",
    "seed": SEED,
    "params": FROZEN_PARAMS,
    "sample_weight": f"n_games clipped to [1, {WEIGHT_CAP}]; reported MAE is unweighted",
    "include_std": bool(INCLUDE_STD),
    "noise_floor_median_elo_range": NOISE_FLOOR,
    "feature_names": list(player_cols),
    "monotone_constraints": monotone_map(player_cols),
    "split_sizes": {name: int((players["split"] == name).sum()) for name in ("train", "val", "test")},
    "allow_list": {
        "numeric_features": list(NUMERIC_FEATURES),
        "engine_features": list(ENGINE_FEATURES),
        "player_extra": list(PLAYER_EXTRA),
        "game_extra": list(GAME_EXTRA),
        "dropped": [
            "eco_family at game level",
            "*_std while median n_games < 20" if not INCLUDE_STD else None,
        ],
    },
    "baselines": {
        "univariate": linear_card(uni_model, [uni_col]),
        "engine_linear": linear_card(engine_model, engine_cols),
    },
    "test_mae": headline["mae"],
    "test_mae_ci95": [headline["mae_ci_lo"], headline["mae_ci_hi"]],
}
player_card["allow_list"]["dropped"] = [item for item in player_card["allow_list"]["dropped"] if item]
game_card = {
    "model": "game",
    "xgb_earned": bool(game_xgb_earned),
    "comparison_baseline": best_game_linear,
    "mae_gap_cluster_ci95": [float(cg_lo), float(cg_hi)],
    "seed": SEED,
    "params": FROZEN_PARAMS,
    "sample_weight": None,
    "feature_names": list(game_feature_names),
    "monotone_constraints": monotone_map(game_feature_names),
    "categoricals": ["first_move"],
    "dropped_features": ["eco_family"],
    "split_row_counts": {name: int(len(prepared[name])) for name in ("train", "val", "test")},
    "allow_list": {
        "numeric_features": list(NUMERIC_FEATURES),
        "engine_features": list(ENGINE_FEATURES),
        "game_extra": list(GAME_EXTRA),
    },
    "baselines": {
        "univariate": linear_card(game_uni, ["mistake_rate"]),
        "engine_linear": linear_card(game_engine, list(ENGINE_FEATURES)),
    },
}
(out_dir / "player_model_card.json").write_text(json.dumps(player_card, indent=2), encoding="utf-8")
(out_dir / "game_model_card.json").write_text(json.dumps(game_card, indent=2), encoding="utf-8")
player_importance.to_csv(out_dir / "player_permutation_importance.csv", index=False)
game_importance.to_csv(out_dir / "game_permutation_importance.csv", index=False)
summary.to_csv(out_dir / "metrics_summary.csv", index=False)
print(f"[summary] saved {out_dir / 'player_xgb.json'} and player_model_card.json")
print(f"[summary] saved {out_dir / 'game_xgb.json'} and game_model_card.json")
print("[summary] saved permutation importances (validation) and metrics_summary.csv")
if Path("/content/drive/MyDrive").exists():
    print(f"[summary] Wrote artifacts to Drive at {out_dir}. They remain after this Colab session ends.")
state_findings(
    headline["mae"], headline["mae_ci_lo"], headline["mae_ci_hi"], headline["spearman"], headline["acc_within_150"],
    gap, gap_lo, gap_hi, best_linear_name, xgb_earned,
    abl_gap, abl_lo, abl_hi, leak,
    NOISE_FLOOR, player_test_bins,
    game_row_metrics["mae"], collapsed_metrics["mae"], game_xgb_earned, float(cg_lo), float(cg_hi),
)
